In [ ]:
from kafka import KafkaConsumer
import json
import base64

consumer = KafkaConsumer(
    "dbserver1.testdb.orders",
    bootstrap_servers="kafka:9092",
    auto_offset_reset="earliest",
    value_deserializer=lambda x: json.loads(x.decode("utf-8"))
)
def decode_amount(value):
    try:
        decoded = base64.b64decode(value)
        return int.from_bytes(decoded, byteorder='big')
    except:
        return 0

for msg in consumer:
    data = msg.value

    payload = data.get("payload")

    print("DEBUG payload:", payload)   

    if payload is None:
        continue

    op = payload.get("op")

    after = payload.get("after")
    print("DEBUG op:", op)   

    if op == "c":
        print("INSERT:", payload.get("after"))

    elif op == "u":
        print("UPDATE:", payload.get("after"))

    elif op == "d":
        print("DELETE:", payload.get("before"))
    if after: 
        # High value alert
        amount = decode_amount(after.get("amount"))
        if amount > 50000:
            print("🚨 HIGH VALUE ORDER ALERT:", after)

        # Status summary
        print(f"[SUMMARY] Order {after['order_id']} is {after['status']}")

        # Customer stream
        print(f"[CUSTOMER STREAM] Customer {after['customer_id']} activity")